# 1. 模型 profile 属性的使用

In [1]:

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek:deepseek-flash",
)

print(model.profile)

{}


# 2. 模型初始化的完整参数
## 2.1 参数打印

In [ ]:
from langchain_openai import ChatOpenAI

# 正确调用方式
print(ChatDeepSeek.model_fields.keys())
print(ChatOpenAI.model_fields.keys())

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-flash",
    base_url=DEEPSEEK_BASE_URL,
    api_key=DEEPSEEK_API_KEY,
)

# 已过时，建议使用上面方式调用
print(model.model_fields.keys())

## 2.2 model_kwargs 参数的使用
model_kwargs 用于存放那些OpenAI Compatible API支持，但 LangChain 没有直接列出的字段，如用于支持
Function Call的 tools 字段。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-flash",
    model_kwargs={"tools": [
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather of a location, the user should supply a location first.",
                "parameters": {
                    "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            }
                        },
                    "required": ["location"]
                },
            }
        },
    ]}
)
# 向模型发送单条数据
response = model.invoke("你好，今天北京的天气如何") # 打印响应
rprint(response)

## 2.2 extra_body 字段
用于存放模型厂商基于OpenAI API协议扩展的字段，比如 thinking 是DeepSeek扩展的字段，用于控制是否启用思考模式，但OpenAI不支持

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(
    model="deepseek:deepseek-flash",
    extra_body={"thinking": {"type": "enabled"}},
)
# 向模型发送单条数据
response = model.invoke("你好，一句话回答") # 打印响应
rprint(response)

# 3. 模型调用中 config 参数的使用
在调用模型时(如使用 invoke(), ainvoke(), stream(),batch()等方法时)，我们可以传入config参数。它允许在调用模型时，动态地配置和控制模型的行为

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1. 初始化模型
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.2,
    max_tokens=500,
    # 指定可调整参数，若后面用户调用时在 configurable 中传入相关参数，则以 configurable 中的参数值为准
    configurable_fields=("model", "model_provider", "temperature", "max_tokens"),
)
# 2. 准备 config 字典
config = {
    "run_name": "joke_generation",     # 在LangSmith中这次运行会显示为
    "joke_generation"
    "tags": ["tag1", "tag2"],          # 打上标签便于分类查找
    "metadata": {"user_id": "123"},    # 记录用户ID
    "configurable":{                   # configurable 中有的配置以 configurable 中的为准，前提 configurable_fields 配置了
        "model": "deepseek-v4-pro",    # 配置模型参数
        "model_provider": "openai",    # 配置模型提供商参数
        "temperature": 0.7,            # 配置温度参数
        "max_tokens": 1000             # 配置最大令牌数
    }
}
# 3. 调用模型并传入config
response = model.invoke(
    "1 + 2 = ?",
    config=config
)


rprint(response)